In [1]:
import os
print(os.getcwd())
print(os.listdir())

/Users/maziarnamdar/ML II
['Task1_Prescription-parser-Comp.ipynb', 'Khash-Task3_News Modeling.ipynb', 'NLP_01_NLP_Pipeline-Solution.ipynb', '16_Videos_Solution.ipynb', 'Text_Classification_Example-Solution.ipynb', 'prescription-parser-model.pkl', 'TextClassification.ipynb', 'Khash_Task4_Movie_Recommenders.ipynb', 'Task1_Prescription-parser-khashNamdar.ipynb', '01_Working_with_Images_Solution.ipynb', 'NLP_01_NLP_Pipeline.ipynb', 'ObjectImage.jpg', 'tags.csv', 'Smart-Chatbot.ipynb', 'ratings.csv', '02_Image_Arithmetics_Solution.ipynb', '01_Working_with_Images.ipynb', 'Chatbot-from-scratch-solution (1).ipynb', 'Khash-Summarization.ipynb', 'article.txt', 'Khash-Object-Segmentation.ipynb', '.ipynb_checkpoints', 'Smart-Chatbot-Solution.ipynb', 'movies.csv', '17_Face_Detection_Solution.ipynb', 'Khash_Text_Classification.txt', 'prescription-parser-model.crfsuite', 'image.jpg', 'newsgroups.json', 'Signature.png', 'Maziar_Task0_Text_Classification.txt']


In [2]:
import sys
import pandas as pd
import numpy as np


DATA_DIR = "/Users/maziarnamdar/Desktop"

movies = pd.read_csv(f"{DATA_DIR}/movies.csv")
ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv")
tags = pd.read_csv(f"{DATA_DIR}/tags.csv")

print(movies.shape, ratings.shape, tags.shape)
movies.head()

(27278, 3) (20000263, 4) (465564, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
ratings.head(5)

,userId,movieId,rating,timestamp
0,1,2,3.5,1112486027
1,1,29,3.5,1112484676
2,1,32,3.5,1112484819
3,1,47,3.5,1112484727
4,1,50,3.5,1112484580


In [4]:
movie_stats = ratings.groupby('movieId').agg(
    v=('rating', 'count'),
    R=('rating', 'mean')
).reset_index()

movie_stats = movie_stats.merge(movies, on='movieId')
movie_stats.head()

,movieId,v,R,title,genres
0,1,49695,3.921240,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,22243,3.211977,Jumanji (1995),Adventure|Children|Fantasy
2,3,12735,3.151040,Grumpier Old Men (1995),Comedy|Romance
3,4,2756,2.861393,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,12161,3.064592,Father of the Bride Part II (1995),Comedy


In [5]:
C = ratings['rating'].mean()
print(C)

m = movie_stats['v'].quantile(0.90)
print(m)

3.5255285642993797
1305.7000000000007


In [6]:
qualified = movie_stats[movie_stats['v'] >= m].copy()

def weighted_rating(row, m=m, C=C):
    v = row['v']
    R = row['R']
    return (v / (v + m) * R) + (m / (v + m) * C)

qualified['score'] = qualified.apply(weighted_rating, axis=1)

qualified = qualified.sort_values('score', ascending=False)
qualified[['title', 'v', 'R', 'score']].head(10)

,title,v,R,score
315,"Shawshank Redemption, The (1994)",63366,4.446990,4.428386
843,"Godfather, The (1972)",41355,4.364732,4.339047
49,"Usual Suspects, The (1995)",47006,4.334372,4.312512
523,Schindler's List (1993),50054,4.310175,4.290227
1195,"Godfather: Part II, The (1974)",27398,4.275641,4.241519
895,Casablanca (1942),24349,4.258327,4.221031
887,Rear Window (1954),17449,4.271334,4.219411
1169,One Flew Over the Cuckoo's Nest (1975),29932,4.248079,4.217877
737,Dr. Strangelove or: How I Learned to Stop Worr...,23220,4.247287,4.208862
2873,Fight Club (1999),40106,4.227123,4.205002


In [7]:
tags['tag'] = tags['tag'].fillna('').astype(str)

tags_grouped = tags.groupby('movieId')['tag'].apply(
    lambda x: ' '.join(x)
).reset_index()

tags_grouped.columns = ['movieId', 'tag_metadata']
tags_grouped.head()

,movieId,tag_metadata
0,1,Watched computer animation Disney animated fea...
1,2,time travel adapted from:book board game child...
2,3,old people that is actually funny sequel fever...
3,4,chick flick revenge characters chick flick cha...
4,5,Diane Keaton family sequel Steve Martin weddin...


In [8]:
content_df = movies.merge(tags_grouped, on='movieId', how='left')
content_df['tag_metadata'] = content_df['tag_metadata'].fillna('')

content_df.head()

,movieId,title,genres,tag_metadata
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Watched computer animation Disney animated fea...
1,2,Jumanji (1995),Adventure|Children|Fantasy,time travel adapted from:book board game child...
2,3,Grumpier Old Men (1995),Comedy|Romance,old people that is actually funny sequel fever...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,chick flick revenge characters chick flick cha...
4,5,Father of the Bride Part II (1995),Comedy,Diane Keaton family sequel Steve Martin weddin...


In [9]:

content_df['genres_clean'] = content_df['genres'].str.replace('|', ' ', regex=False)

content_df['metadata'] = content_df['genres_clean'] + ' ' + content_df['tag_metadata']

content_df[['title', 'metadata']].head()

,title,metadata
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy Wa...
1,Jumanji (1995),Adventure Children Fantasy time travel adapted...
2,Grumpier Old Men (1995),Comedy Romance old people that is actually fun...
3,Waiting to Exhale (1995),Comedy Drama Romance chick flick revenge chara...
4,Father of the Bride Part II (1995),Comedy Diane Keaton family sequel Steve Martin...


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=50000)
tfidf_matrix = tfidf.fit_transform(content_df['metadata'])

tfidf_matrix.shape

(27278, 23865)

In [11]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=200, random_state=42)
latent_matrix_1 = svd.fit_transform(tfidf_matrix)

latent_matrix_1.shape

(27278, 200)

In [12]:
from scipy.sparse import csr_matrix

user_ids = ratings['userId'].astype('category')
movie_ids = ratings['movieId'].astype('category')

user_movie_sparse = csr_matrix(
    (ratings['rating'], (user_ids.cat.codes, movie_ids.cat.codes))
)

user_movie_sparse.shape

(138493, 26744)

In [13]:
svd_user = TruncatedSVD(n_components=200, random_state=42)
latent_matrix_user = svd_user.fit_transform(user_movie_sparse)

latent_matrix_user.shape

(138493, 200)

In [14]:
movie_id_mapping = dict(enumerate(movie_ids.cat.categories))

In [15]:
svd2 = TruncatedSVD(n_components=200, random_state=42)
latent_matrix_2 = svd2.fit_transform(user_movie_sparse.T)

latent_matrix_2.shape

(26744, 200)

In [16]:
# Latent Matrix 1
lm1_df = pd.DataFrame(latent_matrix_1, index=content_df['movieId'].values)

# Latent Matrix 2
lm2_index = [movie_id_mapping[i] for i in range(latent_matrix_2.shape[0])]
lm2_df = pd.DataFrame(latent_matrix_2, index=lm2_index)

# Keep only movies present in both
common_movie_ids = lm1_df.index.intersection(lm2_df.index)
lm1_aligned = lm1_df.loc[common_movie_ids]
lm2_aligned = lm2_df.loc[common_movie_ids]
print(len(common_movie_ids))

26744


In [17]:
from sklearn.metrics.pairwise import cosine_similarity

id_to_title = movies.set_index('movieId')['title'].to_dict()
movieid_to_pos = {mid: pos for pos, mid in enumerate(common_movie_ids)}

In [18]:
def recommend_similar_for_user(user_id, n=10):
    user_ratings = ratings[ratings['userId'] == user_id]
    user_ratings = user_ratings[user_ratings['movieId'].isin(common_movie_ids)]
    
    if user_ratings.empty:
        return f"No rated movies found for user {user_id}."
    
    weights = user_ratings['rating'].values
    
    # take the latent vectors for this user's profile from both content
    # and collaborative matrices (rating-weighted average of what they've rated)
    a_1 = np.average(lm1_aligned.loc[user_ratings['movieId']].values, axis=0, weights=weights).reshape(1, -1)
    a_2 = np.average(lm2_aligned.loc[user_ratings['movieId']].values, axis=0, weights=weights).reshape(1, -1)
    
    # calculate the similarity of this user's profile with all movies
    score_1 = cosine_similarity(lm1_aligned, a_1).reshape(-1)
    score_2 = cosine_similarity(lm2_aligned, a_2).reshape(-1)
    
    # an average measure of both content and collaborative
    hybrid = (score_1 + score_2) / 2.0
    
    dictDf = {'content': score_1, 'collaborative': score_2, 'hybrid': hybrid}
    similar = pd.DataFrame(dictDf, index=common_movie_ids)
    
    # exclude movies the user already rated
    similar = similar[~similar.index.isin(user_ratings['movieId'])]
    
    similar.sort_values('hybrid', ascending=False, inplace=True)
    
    return similar.head(n)

recommend_similar_for_user(1)

,content,collaborative,hybrid
2161,0.557655,0.662671,0.610163
2116,0.568384,0.650372,0.609378
7147,0.499876,0.692096,0.595986
1197,0.530802,0.648318,0.589560
1210,0.442468,0.726120,0.584294
1270,0.408686,0.758103,0.583394
2054,0.409605,0.740162,0.574883
1748,0.432019,0.710425,0.571222
2005,0.438666,0.689785,0.564225
2160,0.479455,0.615866,0.547660


In [19]:
from sklearn.metrics.pairwise import cosine_similarity

user_id_to_idx = {uid: idx for idx, uid in enumerate(user_ids.cat.categories)}

def user_user_recommend(user_id, n=10, k=50):
    if user_id not in user_id_to_idx:
        return f"User {user_id} not found."
    
    target_idx = user_id_to_idx[user_id]
    target_vector = latent_matrix_user[target_idx].reshape(1, -1)
    
    # Similarity between target user and every other user
    sim_scores = cosine_similarity(target_vector, latent_matrix_user)[0]
    
    # Top-k most similar users (excluding the user themself)
    similar_positions = np.argsort(-sim_scores)
    similar_positions = similar_positions[similar_positions != target_idx][:k]
    similar_user_ids = [user_ids.cat.categories[pos] for pos in similar_positions]
    
    # Movies the target user has already rated
    rated_ids = set(ratings[ratings['userId'] == user_id]['movieId'])
    
    # Ratings from similar users, excluding movies target has already seen
    neighbor_ratings = ratings[ratings['userId'].isin(similar_user_ids)]
    neighbor_ratings = neighbor_ratings[~neighbor_ratings['movieId'].isin(rated_ids)]
    
    # Average rating given by similar users, per movie
    movie_scores = neighbor_ratings.groupby('movieId')['rating'].mean().sort_values(ascending=False)
    
    top_ids = movie_scores.head(n).index.tolist()
    return [id_to_title[mid] for mid in top_ids]

user_user_recommend(1)

['Cabin in the Woods, The (2012)',
 'Riding Giants (2004)',
 'What Dreams May Come (1998)',
 "Cat's Meow, The (2002)",
 'Pope of Greenwich Village, The (1984)',
 'Sandlot, The (1993)',
 'Hobbit, The (1977)',
 'Lord of the Flies (1990)',
 'Subway (1985)',
 'Riding in Cars with Boys (2001)']

In [20]:
user_user_recommend(5)

['Fluke (1995)',
 'Raising Arizona (1987)',
 'Kama Sutra: A Tale of Love (1996)',
 'Jungle2Jungle (a.k.a. Jungle 2 Jungle) (1997)',
 'Best Men (1997)',
 'City of Industry (1997)',
 '12 Angry Men (1957)',
 'Boys Life 2 (1997)',
 'Rhyme & Reason (1997)',
 'Grand Day Out with Wallace and Gromit, A (1989)']

In [21]:
import sys
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "scikit-surprise"])

CompletedProcess(args=['/Users/maziarnamdar/anaconda3/envs/notebook/bin/python', '-m', 'pip', 'install', 'scikit-surprise'], returncode=0)

In [22]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

algo = SVD(n_factors=200, random_state=42)
algo.fit(trainset)

predictions = algo.test(testset)
accuracy.rmse(predictions)

RMSE: 0.7871


np.float64(0.7871130593949565)

In [23]:
import time

def recommend_for_user(user_id, n=10):
    rated_movie_ids = set(ratings[ratings['userId'] == user_id]['movieId'])
    unrated_ids = [mid for mid in common_movie_ids if mid not in rated_movie_ids]
    
    predictions = [(mid, algo.predict(user_id, mid).est) for mid in unrated_ids]
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    top_ids = [mid for mid, _ in predictions[:n]]
    return [id_to_title[mid] for mid in top_ids]

start = time.time()
recs = recommend_for_user(1)
elapsed = time.time() - start

print(f"Took {elapsed:.2f} seconds")
recs

Took 0.12 seconds


['Crucified Lovers, The (Chikamatsu monogatari) (1954)',
 'Serenity (2005)',
 'The Imitation Game (2014)',
 'How to Train Your Dragon (2010)',
 'Alone in the Wilderness (2004)',
 'Harry Potter and the Deathly Hallows: Part 1 (2010)',
 'Star Wars: Episode VI - Return of the Jedi (1983)',
 'Frozen Planet (2011)',
 'Jekyll (2007)',
 'I Hired a Contract Killer (1990)']

In [25]:
recommend_similar_for_user(5)

,content,collaborative,hybrid
356,0.465363,0.770984,0.618173
1101,0.408914,0.711210,0.560062
1200,0.476687,0.630862,0.553774
552,0.420781,0.653631,0.537206
1580,0.374596,0.695132,0.534864
1214,0.436921,0.628470,0.532695
3471,0.483901,0.573060,0.528480
527,0.379898,0.667555,0.523727
349,0.366003,0.675832,0.520918
539,0.299470,0.736297,0.517883


In [26]:
recommend_similar_for_user(6)

,content,collaborative,hybrid
761,0.356863,0.696924,0.526893
1210,0.410147,0.640175,0.525161
852,0.264956,0.780364,0.522660
95,0.213483,0.809090,0.511287
1101,0.419634,0.598544,0.509089
1042,0.328247,0.677522,0.502885
673,0.337483,0.649470,0.493476
1393,0.313750,0.672765,0.493257
765,0.288126,0.690534,0.489330
637,0.252038,0.715006,0.483522
